In [1]:
# oh boy. We'll build this off the VLA trainer, though that's a massive mess, but WANDB support is nice.
# If you're reading this, take a break. Go outside and get some fresh air.
# Dealing with this script will make you want to kill yourself.
LAPTOP = True

In [2]:
import os
import shutil

if not os.path.exists("/content/data") and not LAPTOP:
  %mkdir data
  !git clone https://github.com/ggarciabaez/liveleak.git
  %cd liveleak
  %mv modules/stgnn.py ../
  %mv scripts/train_tools.py ../
  %mv scripts/results ../data
  %mv scripts/dataset.py ../
  %cd ../
  %rm -r liveleak
  !pip install --upgrade sentencepiece transformers==5.5.4 tqdm onnx wandb

In [3]:
import os

import wandb
from train_tools import *

# Authenticate to Weights & Biases
# You can set your API key in Colab Secrets (key: 'WANDB_API_KEY') or enter it interactively
if not LAPTOP:
    from google.colab import drive, runtime, userdata
    api_key = userdata.get('WANDB_API_KEY')
    wandb.login(key=api_key)
    drive.mount("/content/drive")
else:
    wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.forge.coreweave.com from /home/ggdev/.netrc.
wandb: Currently logged in as: superfast852 (gg_experiments) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


In [4]:
import math
import time
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.amp import GradScaler
from tqdm import tqdm

from modules.stgnn import STGNN, MotionAnomalyCalibrator

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
CFG = {
    # Paths
    "data_dir": "/home/ggdev/PycharmProjects/liveleak/scripts/results2",
    "episode_glob": "*.npz",
    "checkpoint_dir": "./",

    # Values you're likely to modify
    "epochs": 100,
    "batch_size": 1024,     # tasks per forward pass
    "val_batch_size": 256,      # To get more iterations
    "num_workers": 4,
    "resume": False,
    "overfit_batch": False,

    # Training
    "seed": 42,
    "val_split": 0.1,    # fraction of episodes held out (not tasks)
    "optimizer_stride": 1,      # gradient accumulation over N timesteps
    "grad_clip_norm": 1.0,
    "warmup_epochs": 1.0,
    "learning_rate": 3e-4,
    "backbone_lr_scale": 0.1,
    "weight_decay": 1e-2,
    "log_every": 336,    # optimizer steps between log lines
    "overfit_steps": 500,

    # Model
    "graph_radius": 0.4,
    "hidden_dim": 128,
    "window_size": 9,
    "window_stride": 1,

    # WandB Logging
    "use_wandb": True,
    "wandb_project": "PPE",
    "wandb_entity": None,   # set to your wandb entity/team if applicable
    "wandb_run_name": None,   # optional custom run name
    "wandb_notes": None,
    "wandb_mode": "online", # "online", "offline", or "disabled"
    "watch_model": True,  # whether to track model gradients with wandb.watch
}

In [6]:
import random

from dataset import ContiguousTrajectoryDataset
from torch.nn import functional as F
from torch.utils.data import DataLoader

paths = sorted(Path(CFG["data_dir"]).glob(CFG["episode_glob"]))
random.Random(42).shuffle(paths)
n_val = max(1, int(0.1 * len(paths)))
train_ds = ContiguousTrajectoryDataset(paths[n_val:], window_size=9)   # 8 history + 1 target
val_ds   = ContiguousTrajectoryDataset(paths[:n_val],  window_size=9)

def collate(batch):
    xs, vs = [], []
    for f, v in batch:                         # f: (9,N,13), v: (9,N)
        cone = ((f[..., 4] > 0.5) & v).any(0)  # cone slots
        xs.append(f[:, ~cone]); vs.append(v[:, ~cone])
    N = max(x.shape[1] for x in xs)
    W = xs[0].shape[0]
    X = torch.zeros(len(xs), W, N, 13); X[..., 0] = -1
    V = torch.zeros(len(xs), W, N, dtype=torch.bool)
    for i, (x, v) in enumerate(zip(xs, vs)):
        X[i, :, :x.shape[1]] = x; V[i, :, :x.shape[1]] = v
    return X, V

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  collate_fn=collate, num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=128, shuffle=False, collate_fn=collate)

In [7]:
@torch.no_grad()
def estimate_scale(loader, device, max_batches=200):
    """Std of the true next-step deltas [dpx, dpy, dvx, dvy]; used to balance the 4 loss channels."""
    d = []
    for i, (X, V) in enumerate(loader):
        if i >= max_batches:
            break
        m = V[:, -2] & V[:, -1]
        d.append((X[:, -1, :, 5:9] - X[:, -2, :, 5:9])[m])
    d = torch.cat(d)
    med = d.median(0).values
    return (1.4826 * (d - med).abs().median(0).values).clamp_min(1e-6).to(device)

def batch_loss(model, X, V, scale):
    """X: (B, W, N, 13), V: (B, W, N). History = X[:, :-1], target = last frame.
    Returns (loss, constant_velocity_baseline_loss, n_entities); (None, None, 0) if nothing to score."""
    last = X[:, -2, :, 5:9]                   # [px, py, vx, vy] at last history frame
    tgt = X[:, -1, :, 5:9]                    # target frame
    m = V[:, -2] & V[:, -1]                   # entities present in both
    jump = ((tgt - last).abs() > 8 * scale).any(-1)
    m = m & ~jump
    n = int(m.sum())
    if n == 0:
        return None, None, None, 0
    pred = model(X[:, :-1], V[:, :-1], validate=False)
    loss = F.smooth_l1_loss(pred[m] / scale, tgt[m] / scale)
    with torch.no_grad():
        cv = last.clone()
        cv[..., 0:2] += last[..., 2:4]        # next pos = pos + vel, vel unchanged
        base = F.smooth_l1_loss(cv[m] / scale, tgt[m] / scale)
        stay = F.smooth_l1_loss(last[m] / scale, tgt[m] / scale)

    return loss, base, stay, n

In [8]:
@torch.no_grad()
def fit_calibrator(model, loader, device, path):
    """Calibrate on the per-WINDOW max forecast error (inference scores the max over entities)."""
    model.eval()
    worst = []
    for X, V in loader:
        rmse, ev = model.compute_window_error(X.to(device), V.to(device), return_mask=True)  # (B, N) each
        has = ev.any(1)
        if has.any():
            worst.append(rmse.amax(1)[has].float().cpu())
    if not worst:
        raise RuntimeError("No evaluable windows for calibration")
    cal = MotionAnomalyCalibrator().fit(torch.cat(worst).numpy())
    np.savez(path, errors=cal.normal_errors)
    return cal

In [9]:
def run_epoch(model, loader, optimizer, scheduler, scaler, device, amp_dtype,
              cfg, scale, epoch, global_step, train):
    model.train(train)
    phase = "train" if train else "val"
    opt_stride = cfg.get("optimizer_stride", 1)
    log_every = cfg.get("log_every", 100)
    n_batches = len(loader)

    loss_m, base_m, stay_m, log_m = AverageMeter(), AverageMeter(), AverageMeter(), AverageMeter()
    t0 = time.perf_counter()
    should_step = False
    if train:
        optimizer.zero_grad(set_to_none=True)

    with torch.set_grad_enabled(train):
        for batch_idx, (X, V) in enumerate(tqdm(loader, desc=f"[{phase}] Epoch {epoch}")):
            X = X.to(device, non_blocking=True)
            V = V.to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=device.type == "cuda"):
                loss, base, stay, n = batch_loss(model, X, V, scale)

            if loss is None:                  # e.g. cone-only / empty windows
                continue
            if not torch.isfinite(loss):
                print(f"[{phase}] non-finite loss at epoch {epoch} batch {batch_idx} - skipping")
                if train:
                    optimizer.zero_grad(set_to_none=True)
                continue

            if train:
                scaler.scale(loss / opt_stride).backward()
                should_step = ((batch_idx + 1) % opt_stride == 0) or (batch_idx + 1 == n_batches)
                if should_step:
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), cfg.get("grad_clip_norm", 1.0))
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    if scheduler is not None:
                        scheduler.step()
                    global_step += 1

            loss_m.update(loss.item(), n)
            base_m.update(base.item(), n)
            stay_m.update(stay.item(), n)
            if train:
                log_m.update(loss.item(), n)
        if train and should_step:
            elapsed = max(time.perf_counter() - t0, 1e-6)
            sps = log_every * X.shape[0] * opt_stride / elapsed
            lr = optimizer.param_groups[0]["lr"]
            if cfg.get("use_wandb", True) and wandb.run is not None:
                wandb.log({"train/step_loss": log_m.avg, "train/lr": lr,
                           "train/windows_per_sec": sps, "epoch": epoch}, step=global_step)
            log_m.reset()
    return loss_m.avg, base_m.avg, stay_m.avg, global_step


def train(cfg: dict, run=None) -> None:
    seed_everything(cfg["seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device   : {device}")

    # split by VIDEO (stride-1 windows overlap heavily; a window split would leak)
    paths = sorted(Path(cfg["data_dir"]).glob(cfg.get("episode_glob", "*.npz")))
    if len(paths) < 2:
        raise RuntimeError(f"Need >=2 npz files in {cfg['data_dir']}, found {len(paths)}")
    random.Random(cfg["seed"]).shuffle(paths)
    n_val = max(1, round(cfg["val_split"] * len(paths)))
    val_paths, train_paths = paths[:n_val], paths[n_val:]

    W = cfg["window_size"]                    # total frames = 8 history + 1 target -> 9
    stride = cfg.get("window_stride", 1)
    train_ds = ContiguousTrajectoryDataset(train_paths, window_size=W, window_stride=stride)
    val_ds = ContiguousTrajectoryDataset(val_paths, window_size=W, window_stride=stride)
    print(f"Videos   : {len(train_paths)} train / {len(val_paths)} val   "
          f"Windows: {len(train_ds)} train / {len(val_ds)} val")

    nw = cfg.get("num_workers", 0)
    train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, collate_fn=collate,
                              num_workers=nw, pin_memory=device.type == "cuda")
    val_loader = DataLoader(val_ds, batch_size=cfg.get("val_batch_size", cfg["batch_size"]), shuffle=False,
                            collate_fn=collate, num_workers=nw)

    overfit = cfg.get("overfit_batch", False)
    if overfit:                               # sanity check: loss on ONE batch must go ~0
        one = next(iter(train_loader))
        train_loader = [one] * cfg.get("overfit_steps", 200)
        val_loader = [one]
        cfg = {**cfg, "epochs": 1, "warmup_epochs": 0.05, "log_every": 20}
        print("OVERFIT MODE: 1 batch x", len(train_loader), "steps")

    hidden = cfg.get("hidden_dim", 64)
    model = STGNN(cfg["graph_radius"], hidden).to(device)
    nn.init.zeros_(model.motion_head[-1].weight)
    nn.init.zeros_(model.motion_head[-1].bias)
    scale = estimate_scale(train_loader, device)
    print("Target-delta std [px, py, vx, vy]:", [f"{s:.2e}" for s in scale.tolist()])
    model.cfg = {"graph_radius": cfg["graph_radius"], "hidden_dim": hidden, "window_size": W,
                     "scale": scale.cpu().tolist()}     # save_checkpoint reads model.cfg

    if run is not None and cfg.get("watch_model", False):
        wandb.watch(model, log="gradients", log_freq=cfg.get("log_every", 100))

    optimizer = build_optimizer(model, cfg["learning_rate"], 1.0, cfg["weight_decay"])
    steps_per_epoch = math.ceil(len(train_loader) / cfg.get("optimizer_stride", 1))
    scheduler = build_scheduler(optimizer, cfg, steps_per_epoch)

    bf16_ok = device.type == "cuda" and torch.cuda.is_bf16_supported()
    amp_dtype = torch.bfloat16 if bf16_ok else torch.float16
    scaler = GradScaler("cuda", enabled=(device.type == "cuda" and not bf16_ok))
    print(f"AMP      : {amp_dtype if device.type == 'cuda' else 'off (cpu)'}")

    ckpt_dir = Path(cfg["checkpoint_dir"]); ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_dir / "best.pt"
    start_epoch = global_step = 0
    best_val = float("inf")
    if cfg.get("resume") and best_path.exists():
        start_epoch, global_step, best_val = load_checkpoint(best_path, model, optimizer, scheduler, scaler)
        start_epoch += 1
        print(f"Resumed  : epoch {start_epoch}, step {global_step}, best_val {best_val:.4f}")

    train_loss = val_loss = float("nan")
    for epoch in range(start_epoch, cfg["epochs"]):
        train_loss, _, _, global_step = run_epoch(model, train_loader, optimizer, scheduler, scaler, device,
                                               amp_dtype, cfg, scale, epoch, global_step, train=True)
        val_loss, val_base, val_stay, _ = run_epoch(model, val_loader, None, None, scaler, device,
                                          amp_dtype, cfg, scale, epoch, global_step, train=False)

        is_best = val_loss < best_val
        if is_best:
            best_val = val_loss
            save_checkpoint(best_path, model, optimizer, scheduler, scaler, epoch, global_step, best_val)

        lr = optimizer.param_groups[0]["lr"]
        print(f"epoch {epoch:>3d}  train {train_loss:.4f}  val {val_loss:.4f}  "
              f"const-vel baseline {val_base:.4f}  stay {val_stay:.4f}  ratio {val_loss / max(val_base, 1e-9):.2f}  "
              f"stay ratio {val_loss / max(val_stay, 1e-9):.2f}best {best_val:.4f}  lr {lr:.2e}")

        if cfg.get("use_wandb", True) and wandb.run is not None:
            wandb.log({"train/epoch_loss": train_loss, "val/epoch_loss": val_loss,
                       "val/baseline_loss": val_base, "val/ratio_to_baseline": val_loss / max(val_base, 1e-9),
                       "val/best_loss": best_val, "train/lr": lr, "epoch": epoch}, step=global_step)
            if is_best:
                wandb.run.summary["best_val_loss"] = best_val
                wandb.run.summary["best_epoch"] = epoch

    if cfg.get("use_wandb", True) and wandb.run is not None:
        wandb.run.summary["final_train_loss"] = train_loss
        wandb.run.summary["final_val_loss"] = val_loss
        wandb.run.summary["total_global_steps"] = global_step

    if overfit:
        return

    # matched pair for inference: stgnn.pth + calib.npz (SSOT s5) - from the BEST weights
    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    model.save_weights(ckpt_dir / "stgnn.pth")
    cal = fit_calibrator(model, val_loader, device, ckpt_dir / "calib.npz")
    print(f"Saved stgnn.pth + calib.npz ({cal.normal_errors.size} normal windows) -> {ckpt_dir}")
    return model, scale

In [10]:
if CFG.get("use_wandb", True):
    run = wandb.init(
        project=CFG.get("wandb_project", "VLA"),
        entity=CFG.get("wandb_entity", None),
        name=CFG.get("wandb_run_name", None),
        notes=CFG.get("wandb_notes", None),
        mode=CFG.get("wandb_mode", "online"),
        config=CFG,
        reinit=True,
    )
    try:
        model, scale = train(CFG, run=run)
    finally:
        wandb.finish()
else:
    model, scale = train(CFG, run=None)

import IPython
app = IPython.Application.instance()
app.kernel.do_shutdown(True)

wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.
wandb: setting up run l3kbl4e4
wandb: Tracking run with wandb version 0.31.0
wandb: Run data is saved locally in /home/ggdev/PycharmProjects/liveleak/scripts/wandb/run-20261105_155540-l3kbl4e4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run likely-sun-9
wandb: ⭐️ View project at https://forge.coreweave.com/wandb/gg_experiments/PPE
wandb: 🚀 View run at https://forge.coreweave.com/wandb/gg_experiments/PPE/runs/l3kbl4e4


Device   : cuda
Videos   : 711 train / 79 val   Windows: 107639 train / 12318 val
Target-delta std [px, py, vx, vy]: ['1.00e-04', '2.07e-04', '9.56e-06', '2.05e-05']
AMP      : torch.bfloat16


[val] Epoch 0: 100%|██████████| 49/49 [00:00<00:00, 99.58it/s] 


epoch   0  train 1.1439  val 1.2864  const-vel baseline 0.5704  stay 0.6200  ratio 2.26  stay ratio 2.07best 1.2864  lr 3.00e-04


[val] Epoch 1: 100%|██████████| 49/49 [00:00<00:00, 117.41it/s]


epoch   1  train 1.2699  val 1.3999  const-vel baseline 0.5704  stay 0.6200  ratio 2.45  stay ratio 2.26best 1.2864  lr 3.00e-04


... (196 outputs left out)


wandb: updating run metadata


Saved stgnn.pth + calib.npz (10617 normal windows) -> .


wandb: uploading history steps 98-99, summary
wandb: 
wandb: Run history:
wandb:                 epoch ▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇█████
wandb:      train/epoch_loss ▅▆▆▆▃▄█▃▃▃▃▅▂▂▂▂▂▂▂▂▄▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:              train/lr ████████▇▇▇▇▆▆▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
wandb:       train/step_loss ▆▇▄▃▄▃▃█▃▃▃▃▆▃▃▂▂▂▂▃▃▃▂▂▂▃▁▁▁▂▁▁▁▁▁▁▁▁▁▁
wandb: train/windows_per_sec ▅▅▆▆▇▇▇▆▇▄▇▇██▆▆▆▄▆▇▇▄▅▄▇▇▅▇▅▅▆▁▆▆▅▅▇▆▆▆
wandb:     val/baseline_loss ▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:         val/best_loss █▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb:        val/epoch_loss ▇█▄▃▂▃▂▃▃▃▃▅▃▂▂▂▂▂▂▂▃▂▃▂▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: val/ratio_to_baseline ▄▅▂█▃▂▂▂▂▃▄▂▂▂▂▂▂▂▂▁▃▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁
wandb: 
wandb: Run summary:
wandb:            best_epoch 96
wandb:         best_val_loss 0.62003
wandb:                 epoch 99
wandb:      final_train_loss 0.71129
wandb:        final_val_loss 0.62005
wandb:    total_global_steps 10600
wandb:      train/epoch_loss 0.71129
wandb:              t

{'status': 'ok', 'restart': True}

In [11]:
if not LAPTOP:
    os.makedirs("/content/drive/MyDrive/liveleak/checkpoints/model_v1", exist_ok=True)
    shutil.copytree(CFG["checkpoint_dir"], "/content/drive/MyDrive/liveleak/checkpoints/model_v1", dirs_exist_ok=True)
    time.sleep(120)  # wait 5m for everything to finish up
    runtime.unassign()

In [12]:
x = np.load(paths[0], allow_pickle=False)
x["metadata"].item()

'{"source_video": "datasets", "width": 1920, "height": 1080, "src_fps": 25.000000000000004, "vid_stride": 1, "effective_fps": 25.000000000000004, "num_frames": 201, "total_rows": 606, "config": {"model_path": "../assets/ppe_50ep.pt", "conf": 0.25, "target_fps": 30.0, "tracker": {"type": "OCSORTTracker", "track_activation_threshold": 0.25, "minimum_matching_threshold": 0.8, "minimum_consecutive_frames": 1, "track_life_seconds": 2.0}}}'

In [ ]:
import json

def file_stats(path):
    d = np.load(path, allow_pickle=False); rows, ptr = d["rows"], d["frame_ptr"]
    meta = json.loads(d["metadata"].item()); dp = []
    for t in range(len(ptr) - 2):
        a, b = rows[ptr[t]:ptr[t+1]], rows[ptr[t+1]:ptr[t+2]]
        a, b = a[a[:, 0] > -1], b[b[:, 0] > -1]
        _, ia, ib = np.intersect1d(a[:, 0], b[:, 0], return_indices=True)
        dp.append(np.hypot(b[ib, 5] - a[ia, 5], b[ib, 6] - a[ia, 6]))
    ad = np.concatenate(dp); p = np.percentile(ad, [50, 90, 99, 99.9])
    print(f"{Path(path).name[:22]:22s} {meta['width']}x{meta['height']} n={ad.size:7d} "
          f"p50/90/99/99.9={p[0]:.1f}/{p[1]:.1f}/{p[2]:.1f}/{p[3]:.1f} max={ad.max():.0f} >40px={np.mean(ad>40):.2%}")
for p in paths: file_stats(p)

In [ ]:
@torch.no_grad()
def diagnose(model, loader, scale, device, min_hist=1, radius=None):
    old = model.graph_radius
    if radius is not None: model.graph_radius = radius
    model.eval()
    acc = {k: torch.zeros(4, device=device) for k in ("model", "const_vel", "stay")}; n = 0
    for X, V in loader:
        X, V = X.to(device), V.to(device)
        m = V[:, -2] & V[:, -1]
        if min_hist >= 2: m = m & V[:, -3]
        if not m.any(): continue
        last, tgt = X[:, -2, :, 5:9], X[:, -1, :, 5:9]
        pred = model(X[:, :-1], V[:, :-1], validate=False)
        cv = last.clone(); cv[..., :2] += last[..., 2:]
        for k, p in (("model", pred), ("const_vel", cv), ("stay", last)):
            acc[k] += ((p - tgt)[m] / scale).abs().sum(0)
        n += int(m.sum())
    model.graph_radius = old
    for k, v in acc.items():
        print(f"{k:10s} MAE/std [px,py,vx,vy]:", [f"{x:.2f}" for x in (v / n).tolist()])

device = DEVICE
for name, ld in (("train", train_loader), ("val", val_loader)): print(name); diagnose(model, ld, scale, device)
print("val, tracks with >=2 history frames"); diagnose(model, val_loader, scale, device, min_hist=2)
print("val, NO EDGES");                       diagnose(model, val_loader, scale, device, radius=1e-4)

In [ ]:
from glob import glob
from pathlib import Path

import numpy as np

# paths = sorted(Path(CFG["data_dir"]).glob(CFG["episode_glob"]))
paths = sorted(glob("results2/*.npz"))
path = paths[5]
d = np.load(path, allow_pickle=False); rows, ptr = d["rows"], d["frame_ptr"]
dp, v0, v1 = [], [], []
for t in range(len(ptr) - 2):
    a, b = rows[ptr[t]:ptr[t+1]], rows[ptr[t+1]:ptr[t+2]]
    a, b = a[a[:, 0] > -1], b[b[:, 0] > -1]                    # skip cones
    _, ia, ib = np.intersect1d(a[:, 0], b[:, 0], return_indices=True)
    dp.append(b[ib, 5] - a[ia, 5]); v0.append(a[ia, 7]); v1.append(b[ib, 7])
dp, v0, v1 = map(np.concatenate, (dp, v0, v1))
print("corr/slope dpx~v_t  :", np.corrcoef(dp, v0)[0, 1], np.polyfit(v0, dp, 1)[0])
print("corr/slope dpx~v_t+1:", np.corrcoef(dp, v1)[0, 1], np.polyfit(v1, dp, 1)[0])
print("std dpx / std vx    :", dp.std(), v0.std())